# HIVE lesion training: folds 0–4 (single GPU, sequential)

This notebook trains the five inference-compatible T2+ADC+HIGHB lesion folds from the current deployed fold weights. It uses the full HIVE inference `output/` tree, never `Interesting outputs/`.

Safety and data policy:

- One notebook cell trains folds 0, 1, 2, 3, and 4 automatically in sequence. Folds never run concurrently. Every container is restricted to physical GPU 0 and must see exactly one CUDA device.
- A lock prevents concurrent organ/lesion notebook jobs on GPU 0.
- All HIVE outputs are scanned, but an SGH lesion label is admitted only when `Reviewed=Y`, `Q2=Y`, `Q3=Y`, and `Q4=Y`, followed by technical QC. A missing lesion mask is never interpreted as a negative label.
- Fold-specific development splits are locked; the held-out test split is untouched during selection.
- `model_last_foldN.pth.tar` is updated every epoch, a better validation Dice updates `model_best_foldN.pth.tar`, and every 50th epoch gets a permanent full snapshot in the run's `checkpoints/` directory.


In [ ]:
from pathlib import Path
import os
import csv
import json
import re
import shlex
import subprocess

PROJECT = Path.cwd().resolve()
HIVE_RUN = Path(os.environ.get(
    'PROSTATE_HIVE_RUN',
    '/home/ddsadmin/Documents/Prostate HIVE Inference 4 jun 2026 - highb fix',
)).expanduser().resolve()
TRAINING_IMAGE = 'prostate-mri-training:latest'
GPU_ID = 0

HELPER = PROJECT / 'scripts' / 'hive_training.py'
assert HELPER.is_file(), f'Open this notebook with {PROJECT} as the working directory'

BASE = [
    'python3', str(HELPER),
    '--project', str(PROJECT),
    '--hive-run', str(HIVE_RUN),
    '--image', TRAINING_IMAGE,
    '--gpu-id', str(GPU_ID),
]

def run_helper(*arguments):
    command = [*BASE, *map(str, arguments)]
    print('$', shlex.join(command), flush=True)
    subprocess.run(command, cwd=PROJECT, check=True)

print('Project:', PROJECT)
print('SGH source:', HIVE_RUN / 'output')
print('Transfer weights:', HIVE_RUN / 'models')
print('Physical GPU:', GPU_ID)


## 1. Mandatory preflight

Run this before starting or resuming any fold. It verifies that physical GPU 0 is idle, exactly one GPU is visible inside the container, strict-loads the current organ plus all five lesion transfer weights, and rejects an unsafe or internally inconsistent YAML configuration. If GPU 0 is already occupied by inference or another training job, preflight fails closed.

In [ ]:
run_helper('preflight')

## 2. Inspect the already-prepared locked lesion folds

If audit/source data changed, rebuild it from the organ notebook first. The persistent preprocessing cache at `artifacts/training/lesion_cache/` is shared safely across folds and can be large.

In [ ]:
report_path = PROJECT / 'artifacts/training/audit_report.json'
assert report_path.is_file(), 'Prepare the cohort from hive_train_organ.ipynb first.'
report = json.loads(report_path.read_text())
print('Lesion-eligible cases:', report['lesion_eligible'])

with (PROJECT / 'artifacts/training/lesion_splits.csv').open(newline='') as handle:
    rows = list(csv.DictReader(handle))
test_count = sum(row['split'] == 'test' for row in rows)
development = [row for row in rows if row['split'] == 'development']
print('Held-out test cases (do not tune on these):', test_count)
for fold in range(5):
    val_count = sum(row['fold'] == str(fold) for row in development)
    print(f'fold {fold}: train={len(development) - val_count}, val={val_count}')

## 3. Train folds 0 → 4 automatically and sequentially

Set `START_ALL_FOLDS=True` once. The cell blocks while fold 0 trains, then launches folds 1, 2, 3, and 4 in order. Each helper call must finish before the next starts, so only one training container and one physical GPU are used at any moment. Any failed fold raises an error and stops the sequence; later folds are not launched.

Completed folds matching the current YAML epoch count are skipped when `SKIP_COMPLETED_FOLDS=True`. An incomplete fold automatically resumes from its newest compatible `model_last_foldN.pth.tar` when `AUTO_RESUME_INCOMPLETE=True`. You may override a resume path explicitly in `RESUME_BY_FOLD`. Resume and initial weights are mutually exclusive.

The epoch count is read directly from `configs/lesion.yaml` so the notebook and YAML cannot silently disagree. The completed HIVE configuration uses 50 epochs, a conservative `1e-4` learning rate, effective batch size 2, validation every epoch, last-state recovery every epoch, and permanent snapshots every 10 epochs. Do not start the GPU inference app while this cell is running.

In [ ]:
START_ALL_FOLDS = False
FOLDS_TO_TRAIN = list(range(5))  # [0, 1, 2, 3, 4], always sequential.
SKIP_COMPLETED_FOLDS = True
AUTO_RESUME_INCOMPLETE = True
RESUME_BY_FOLD = {
    # 0: 'experiments/lesion/fold0/<run>/model_last_fold0.pth.tar',
}

config_text = (PROJECT / 'configs/lesion.yaml').read_text()
epoch_match = re.search(r'(?m)^  epochs:\s*(\d+)\s*$', config_text)
assert epoch_match, 'Could not read train.epochs from configs/lesion.yaml'
LESION_EPOCHS = int(epoch_match.group(1))
assert LESION_EPOCHS >= 1, f'Invalid epoch count: {LESION_EPOCHS}'
assert FOLDS_TO_TRAIN == sorted(set(FOLDS_TO_TRAIN)), 'Folds must be unique and increasing.'
assert all(fold in range(5) for fold in FOLDS_TO_TRAIN)

def current_completed_runs(fold):
    runs = sorted((PROJECT / f'experiments/lesion/fold{fold}').glob('*'))
    return [run for run in runs
            if (run / 'run_summary.json').is_file()
            and f'  epochs: {LESION_EPOCHS}\n' in (run / 'resolved_config.yaml').read_text()]

def newest_incomplete_checkpoint(fold):
    runs = sorted((PROJECT / f'experiments/lesion/fold{fold}').glob('*'), reverse=True)
    for run in runs:
        checkpoint = run / f'model_last_fold{fold}.pth.tar'
        resolved = run / 'resolved_config.yaml'
        compatible = resolved.is_file() and f'  epochs: {LESION_EPOCHS}\n' in resolved.read_text()
        if compatible and not (run / 'run_summary.json').is_file() and checkpoint.is_file():
            return checkpoint
    return None

if START_ALL_FOLDS:
    for position, fold in enumerate(FOLDS_TO_TRAIN, start=1):
        print(f'\n=== Sequential lesion fold {fold} ({position}/{len(FOLDS_TO_TRAIN)}) ===', flush=True)
        completed = current_completed_runs(fold)
        if SKIP_COMPLETED_FOLDS and completed:
            print(f'Skipping fold {fold}; completed run: {completed[-1]}', flush=True)
            continue
        resume = RESUME_BY_FOLD.get(fold, '')
        resume_path = (PROJECT / resume).resolve() if resume else None
        if resume_path is None and AUTO_RESUME_INCOMPLETE:
            resume_path = newest_incomplete_checkpoint(fold)
        if resume_path:
            print(f'Resuming fold {fold} from {resume_path}', flush=True)
            run_helper('lesion', '--fold', fold, '--resume', resume_path)
        else:
            run_helper('lesion', '--fold', fold)
        run_helper('status')
    print('\nAll requested folds completed or were safely skipped.', flush=True)
else:
    print('Not started. Run preflight, then set START_ALL_FOLDS=True once.')

## 4. Track fold progress

Run these cells at any time. An incomplete run still has a last checkpoint after every finished epoch. Do not choose a model by training loss; the stable best checkpoint is selected by validation Dice. Each fold's experiment record, metrics, recovery checkpoint, best checkpoint, and periodic snapshots remain under `experiments/lesion/foldN/<run-id>/`.

In [ ]:
run_helper('status')

In [ ]:
for fold in range(5):
    metrics_files = sorted((PROJECT / f'experiments/lesion/fold{fold}').glob('*/metrics.csv'))
    if not metrics_files:
        print(f'fold {fold}: no run yet')
        continue
    metrics_path = metrics_files[-1]
    with metrics_path.open(newline='') as handle:
        metrics = list(csv.DictReader(handle))
    scored = [row for row in metrics if row.get('val_dice')]
    best = max(scored, key=lambda row: float(row['val_dice'])) if scored else None
    complete = (metrics_path.parent / 'run_summary.json').is_file()
    print(f'fold {fold}: run={metrics_path.parent.name} last_epoch={metrics[-1]["epoch"] if metrics else "none"} complete={complete}')
    print('  best validation row:', best)
    print('  recovery:', metrics_path.parent / f'model_last_fold{fold}.pth.tar')
    print('  permanent snapshots:', sorted((metrics_path.parent / 'checkpoints').glob('*')))

## 5. Create the inference-ready export bundle

Run only after one completed organ run and all five completed lesion folds exist. The helper selects the highest validation Dice among completed runs for each model, copies the unchanged classifier, and writes SHA-256 checksums. The new bundle is placed under `training_exports/` and can replace the inference `models/` mount only after final review and locked-test evaluation.

In [ ]:
CREATE_EXPORT_BUNDLE = False

if CREATE_EXPORT_BUNDLE:
    run_helper('package')
else:
    print('Not packaged. Set CREATE_EXPORT_BUNDLE=True after organ + folds 0–4 complete.')

## 6. Final evaluation rule

Do not repeatedly inspect the locked test cohort. Freeze the organ and all five lesion choices first, create the bundle, and then run the repository's documented `training.evaluate` commands once. Keep the original pretrained bundle and the new checksum manifest so rollback is immediate.